# Vodafone Music Subscription Prediction

## Sprint 1 - Data Cleaning & Preparation

Goal of this notebook:

Prepare a clean and model-ready dataset by handling missing values,
removing low-quality features, and eliminating technical columns.

### 1. Cleaning Strategy

Based on the EDA findings:

- The dataset contains high dimensionality (461 features)
- 62 features contain more than 50% missing values
- Strong class imbalance is present
- Several features show sparse behavioral patterns

Cleaning approach:

1. Remove non-informative technical columns (e.g., id)
2. Interpret missing values in behavioral features (count_*, vol_\*, \*\_flag\_\*) as absence of activity and impute with 0
3. Apply median imputation to remaining numerical features
4. Validate that no missing values remain
5. Save the cleaned dataset separately

### 2. Load Data

We load the raw training dataset and remove technical identifiers that should not be used for learning.

In [1]:
import pandas as pd
import numpy as np

DATA_PATH = "../data/raw/music.csv"
DICT_PATH = "../data/raw/Feature_dictionary.xlsx"

df = pd.read_csv(DATA_PATH)
print("Original shape:", df.shape)

# Drop technical identifier
if "id" in df.columns:
    df = df.drop(columns=["id"])

print("After dropping id:", df.shape)

Original shape: (70000, 461)
After dropping id: (70000, 460)


### 3. Feature Dictionary Mapping (Feature → Group)

We build a mapping from each feature name to its semantic group (13 groups) using `Feature_dictionary.xlsx`.
This allows us to apply different imputation rules depending on feature meaning.

In [2]:
fd = pd.read_excel(DICT_PATH)

feature_to_group = {}
current_group = None

for _, row in fd.iterrows():
    name = row["Feature Name"]
    desc = row["Feature Description"]

    if pd.isna(name):
        continue

    name = str(name).strip()

    if pd.isna(desc):
        current_group = name
        continue

    if current_group:
        feature_to_group[name.lower()] = current_group

print("Mapped features from dictionary:", len(feature_to_group))

def strip_m_suffix(col: str) -> str:
    col = col.lower()
    return col.replace("_m1", "").replace("_m2", "").replace("_m3", "")


Mapped features from dictionary: 221


#### Mapping Result Summary

The feature dictionary mapping successfully identified 221 base features and assigned them to one of the 13 semantic groups defined in Feature_dictionary.xlsx.

Dynamic features with suffixes (_m1, _m2, _m3) are handled via base-name matching using the strip_m_suffix() function.
This ensures that time-based versions of the same feature inherit the same semantic group.

This mapping enables:

* Group-based imputation rules

* Semantically consistent data cleaning

* Structured feature engineering

* Better interpretability of modeling decisions

The dictionary coverage was validated before proceeding to the cleaning stage.

### 4. Dictionary Coverage (Mapped vs Unmapped)

We verify how many columns are covered by the dictionary (direct match or via `_m1/_m2/_m3` suffix stripping).

In [3]:
df_cols = [c for c in df.columns if c != "target"]

direct_mapped = [c for c in df_cols if c in feature_to_group]

mapped_via_suffix = []
unmapped = []

for c in df_cols:
    if c in feature_to_group:
        continue
    base = strip_m_suffix(c)
    if base in feature_to_group:
        mapped_via_suffix.append(c)
    else:
        unmapped.append(c)

print("\n--- Feature Dictionary Coverage ---")
print(f"Total columns (excluding target): {len(df_cols)}")
print(f"Mapped directly: {len(direct_mapped)}")
print(f"Mapped via _m1/_m2/_m3 base name: {len(mapped_via_suffix)}")
print(f"Unmapped (fallback rules will apply): {len(unmapped)}")

if unmapped:
    print("Unmapped examples:", unmapped[:20])


--- Feature Dictionary Coverage ---
Total columns (excluding target): 459
Mapped directly: 102
Mapped via _m1/_m2/_m3 base name: 357
Unmapped (fallback rules will apply): 0


#### Coverage Interpretation

All dataset features are successfully mapped to one of the 13 semantic groups.

* 102 features are directly defined in the dictionary.

* 357 dynamic features (`_m1`, `_m2`, `_m3`) correctly inherit their base feature group.

* No unmapped features were detected.

This confirms that semantic grouping is complete and group-based imputation can be applied safely.

### 5. Missing Values Audit

We compute missing ratios and summarize the overall missingness in the dataset.

In [4]:
missing_ratio = df.isna().mean().sort_values(ascending=False)

print("Top-20 missing ratio:")
print(missing_ratio.head(20))

total_missing = int(df.isna().sum().sum())
cols_with_missing = int((df.isna().sum() > 0).sum())

print("\nTotal missing values:", total_missing)
print("Columns with >=1 missing:", cols_with_missing)

Top-20 missing ratio:
count_url_category_13    0.992771
count_url_category_15    0.987029
vol_app_3                0.986643
count_app_3              0.986643
count_url_category_12    0.983129
vol_app_11               0.977329
count_app_11             0.977329
count_app_13             0.973143
vol_app_13               0.973143
count_url_category_1     0.972186
count_gift_type_4        0.969586
rr_gift_type_4           0.969586
count_app_16             0.967171
vol_app_16               0.967171
vol_app_14               0.964086
count_app_14             0.964086
count_url_category_11    0.948486
service_9_flag_m3        0.947871
service_9_flag_m1        0.945414
service_9_flag_m2        0.945157
dtype: float64

Total missing values: 5113163
Columns with >=1 missing: 449


#### Missingness Interpretation

The dataset exhibits a very high level of structural sparsity:

* 449 out of 459 features contain missing values.

* Over 5 million missing entries are present in total.

* The most sparse features (>95% missing) belong primarily to:

  * Interests Data

  * App Usage

  * Campaign Activity

This pattern suggests that missing values are not random noise but reflect absence of user activity.

Therefore, missingness will be treated as meaningful signal rather than purely data quality issue.

### 6. Missingness as Signal (Indicators)

For the top sparse features, we add binary indicators `__was_missing` and check their correlation with `target`.
This step is used as an argument/interpretability check (not as a final feature selection rule).

In [5]:
TOP_N = 20
# Remove existing missing indicators if block was executed before
df = df.loc[:, ~df.columns.str.endswith("__was_missing")]

top_sparse = [c for c in missing_ratio.head(TOP_N).index.tolist() if c != "target"]

miss_df = df[top_sparse].isna().astype(int)
miss_df.columns = [f"{c}__was_missing" for c in top_sparse]

df = pd.concat([df, miss_df], axis=1).copy()

if "target" in df.columns:
    miss_corr = df[miss_df.columns.tolist() + ["target"]].corr(numeric_only=True)["target"].sort_values(ascending=False)

    print("\nTop missingness indicators correlation with target:")
    print(miss_corr.head(10))

    print("\nBottom missingness indicators correlation with target:")
    print(miss_corr.tail(10))


Top missingness indicators correlation with target:
target                                1.000000
rr_gift_type_4__was_missing           0.028371
count_gift_type_4__was_missing        0.028371
count_url_category_15__was_missing    0.005136
count_url_category_11__was_missing   -0.000714
vol_app_11__was_missing              -0.001023
count_app_11__was_missing            -0.001023
count_url_category_13__was_missing   -0.001072
count_url_category_1__was_missing    -0.003795
vol_app_16__was_missing              -0.005750
Name: target, dtype: float64

Bottom missingness indicators correlation with target:
count_app_13__was_missing            -0.006557
vol_app_13__was_missing              -0.006557
count_url_category_12__was_missing   -0.008203
vol_app_14__was_missing              -0.010046
count_app_14__was_missing            -0.010046
vol_app_3__was_missing               -0.015613
count_app_3__was_missing             -0.015613
service_9_flag_m3__was_missing       -0.096357
service_9_flag_m

#### Missingness Signal Interpretation

Missingness indicators demonstrate generally weak correlation with the target variable.

* Most correlations are close to zero.

* The strongest negative correlations (~ -0.13) are observed for `service_9_flag` features.

* No strong predictive patterns are identified.

This confirms that missing values largely represent absence of activity rather than structured segmentation.

Therefore, missingness is treated primarily as structural sparsity, and behavioral features will be imputed with zeros.

Indicators are retained as optional features for potential modeling benefit.

### 7. Group-Based Imputation Rules

Imputation rules depend on feature meaning (group from the dictionary):

- Activity / usage metrics (cost/count/duration/volume, campaigns, interests, etc.) → `0`
- User profile-like continuous metrics (e.g., balance/inactive days) → `median`
- Device-related categorical codes → `mode`
- Unmapped features → conservative fallback (`0` for activity-like, otherwise `median`)

In [6]:
def norm_group(s: str) -> str:
    return " ".join(str(s).strip().lower().split())

# Normalize dictionary group values once
feature_to_group_norm = {k: norm_group(v) for k, v in feature_to_group.items()}

# Define 13 groups (normalized)
G_DEVICE  = norm_group("Device Characteristics")
G_GENERAL = norm_group("General Characteristics")
G_UACT    = norm_group("User Activity Features")
G_DYN     = norm_group("Features in Dynamics (Previous month, 2 Months ago, 3 Month Ago with prfx m1,m2,m3)")
G_COSTS   = norm_group("All Costs")
G_RECH    = norm_group("Recharg. Statistics")
G_VCOST   = norm_group("Voice Costs")
G_VDUR    = norm_group("Voice Duration")
G_NET     = norm_group("Network Actions Stats")
G_ESS     = norm_group("Other Essential Features")
G_SMS     = norm_group("SMS from Services")
G_INTER   = norm_group("Interests Data")
G_CAMP    = norm_group("Previous Campaigns Results")

zero_groups   = {G_COSTS, G_RECH, G_VCOST, G_VDUR, G_NET, G_SMS, G_INTER, G_CAMP}
median_groups = {G_UACT}

def fill_mode(series: pd.Series):
    m = series.mode(dropna=True)
    return m.iloc[0] if len(m) else 0

report = []

for col in df.columns:
    if col == "target":
        continue
    if df[col].isna().sum() == 0:
        continue

    # Determine group (exact or via base name)
    grp = None
    if col in feature_to_group_norm:
        grp = feature_to_group_norm[col]
    else:
        base = strip_m_suffix(col)
        if base in feature_to_group_norm:
            grp = feature_to_group_norm[base]

    n_missing = int(df[col].isna().sum())

    if grp in zero_groups:
        df[col] = df[col].fillna(0)
        report.append((col, grp, n_missing, "fillna(0)"))

    elif grp in median_groups:
        df[col] = df[col].fillna(df[col].median())
        report.append((col, grp, n_missing, "median"))

    elif grp == G_DEVICE:
        if col.lower() == "sim_count":
            df[col] = df[col].fillna(df[col].median())
            report.append((col, grp, n_missing, "median"))
        else:
            df[col] = df[col].fillna(fill_mode(df[col]))
            report.append((col, grp, n_missing, "mode"))

    elif grp == G_GENERAL:
        cl = col.lower()
        if "flag" in cl or "count" in cl:
            df[col] = df[col].fillna(0)
            report.append((col, grp, n_missing, "fillna(0)"))
        elif cl in ["lt", "days_exp"]:   # <-- FIX
            df[col] = df[col].fillna(df[col].median())
            report.append((col, grp, n_missing, "median"))
        else:
            df[col] = df[col].fillna(df[col].median())
            report.append((col, grp, n_missing, "median(fallback)"))

    elif grp == G_DYN:
        cl = col.lower()
        if any(k in cl for k in ["count", "cost", "dur", "flag", "vol", "sum", "paym", "sms", "voice", "data", "content"]):
            df[col] = df[col].fillna(0)
            report.append((col, grp, n_missing, "fillna(0)"))
        else:
            df[col] = df[col].fillna(df[col].median())
            report.append((col, grp, n_missing, "median"))

    elif grp == G_ESS:
        cl = col.lower()
        if cl.startswith("income"):
            df[col] = df[col].fillna(df[col].median())
            report.append((col, grp, n_missing, "median(income)"))
        elif any(k in cl for k in ["count", "cost", "flag", "content", "service", "data_type", "vol"]):
            df[col] = df[col].fillna(0)
            report.append((col, grp, n_missing, "fillna(0)"))
        else:
            df[col] = df[col].fillna(df[col].median())
            report.append((col, grp, n_missing, "median(fallback)"))

    else:
        cl = col.lower()
        if any(k in cl for k in ["count", "cost", "dur", "flag", "vol", "sms", "voice", "paym", "content", "data_type"]):
            df[col] = df[col].fillna(0)
            report.append((col, "unmapped", n_missing, "fillna(0) fallback"))
        else:
            df[col] = df[col].fillna(df[col].median())
            report.append((col, "unmapped", n_missing, "median fallback"))

rep_df = pd.DataFrame(report, columns=["feature", "group", "missing_filled", "method"])
print("\nImputation methods used:")
print(rep_df["method"].value_counts())


Imputation methods used:
method
fillna(0)         441
median              5
median(income)      3
Name: count, dtype: int64


#### Imputation Summary and Rationale

The imputation process was performed using group-based rules derived from the feature dictionary.

Key observations:

* The majority of features represent activity-based metrics (usage, costs, counts, durations, campaign interactions, interest data).

* For such features, missing values naturally indicate absence of activity and were therefore imputed with 0.

* A small subset of profile-related continuous variables (e.g., lifetime, balance, inactivity days, expiration days) was imputed using the median, as zero would introduce unrealistic values.

* Income-related features were also imputed using the median to preserve distributional properties.

Imputation distribution:

* Activity features filled with 0 - dominant strategy

* Profile-like continuous features - filled with median

* No remaining missing values after imputation

### 8. Imputation Report (Samples)

We keep a small report of how missing values were filled.

In [7]:
display(rep_df.head(20))

,feature,group,missing_filled,method
0,sim_count,device characteristics,699,median
1,days_exp,general characteristics,7,median
2,service_1_flag,general characteristics,7,fillna(0)
3,service_1_count,general characteristics,7,fillna(0)
4,service_2_flag,general characteristics,7,fillna(0)
5,service_3_flag,general characteristics,7,fillna(0)
6,balance_sum,user activity features,7,median
7,paym_last_days,user activity features,829,median
8,inact_days_count,user activity features,7,median
9,count_sms_source_1,sms from services,7934,fillna(0)


#### Imputation Report (Samples)

The table above presents a sample of features with missing values and the applied imputation strategy.

Observations:

* Device Characteristics

  * `sim_count` was imputed using the median, as zero would be unrealistic.

* General Characteristics

  * Binary flags (`service_1_flag`, `service_2_flag`, etc.) were imputed with 0, assuming absence of the service.

  * Continuous variables like `days_exp` were imputed with the median to preserve distribution stability.

* User Activity Features

  * Behavioral intensity metrics (`balance_sum`, `paym_last_days`, `inact_days_count`) were imputed using the median, as they represent continuous client behavior rather than event counts.

* SMS from Services

All count-based features were filled with 0, since missing values indicate no recorded activity.

This confirms that imputation decisions were applied consistently and in accordance with feature semantics rather than using a single global rule.

No missing values remain in the dataset after this stage.

### 9. Low-Information Features Removal

We remove:
- Constant features (`nunique <= 1`)
- Quasi-constant features (top value share > 99.9%)

In [9]:
# Constant columns
constant_cols = [c for c in df.columns if c != "target" and df[c].nunique(dropna=False) <= 1]
df = df.drop(columns=constant_cols)
print("Constant columns removed:", len(constant_cols))

# Quasi-constant columns
threshold = 0.999
quasi_constant_cols = []
for col in [c for c in df.columns if c != "target"]:
    top_share = df[col].value_counts(normalize=True, dropna=False).iloc[0]
    if top_share > threshold:
        quasi_constant_cols.append(col)

df = df.drop(columns=quasi_constant_cols)
print("Quasi-constant columns removed (>99.9% same):", len(quasi_constant_cols))

Constant columns removed: 17
Quasi-constant columns removed (>99.9% same): 20


After removing constant and quasi-constant features:

- 17 constant features were removed (no variance).
- 20 quasi-constant features were removed (>99.9% identical values).

These features provide no predictive power and may negatively impact model stability.

The dataset is now reduced in dimensionality while preserving informative variability.

### 10. Duplicate Check

We explicitly check and remove duplicate rows (if any).

In [10]:
dup_count = int(df.duplicated().sum())
print("Duplicate rows BEFORE drop:", dup_count)

if dup_count > 0:
    df = df.drop_duplicates().copy()

print("Duplicate rows AFTER drop:", int(df.duplicated().sum()))

Duplicate rows BEFORE drop: 0
Duplicate rows AFTER drop: 0


No duplicate rows were found in the dataset.

Each observation represents a unique customer profile, and no redundant records were removed.


### 11. Final Validation & Save

In [11]:
print("\n--- Final Validation ---")
print("Final shape:", df.shape)
print("Remaining NaNs:", int(df.isna().sum().sum()))
print("Columns with NaNs:", int((df.isna().sum() > 0).sum()))

out_path = "../data/processed/train_cleaned.csv"
df.to_csv(out_path, index=False)
print(f"Saved cleaned dataset to: {out_path}")


--- Final Validation ---
Final shape: (70000, 449)
Remaining NaNs: 0
Columns with NaNs: 0
Saved cleaned dataset to: ../data/processed/train_cleaned.csv


#### Final Dataset Status

After applying group-based imputation and removing low-information features:

- Final dataset shape: **70,000 rows × 449 columns**
- Missing values remaining: **0**
- Columns containing missing values: **0**

The cleaned dataset was saved to:

`data/processed/train_cleaned.csv`

This file will be used as the primary input for preprocessing and baseline model training.
